# 21. Baseline CNN pada data Palm bersih (AlexNet dan MobileNetV2)
**Penelitian:** Deteksi anemia dari citra pucat (telapak tangan, kuku, konjungtiva) dengan data pooling multi-situs.

**Tujuan:** melatih AlexNet dan MobileNetV2 pada data Palm bersih untuk skema 70:10:20 dan 80:10:10 dengan split yang sama dengan Notebook 20 (seed 42). AlexNet mengikuti Appiahene et al. (2023): SGD, ReLU, regularisasi L2 0,0001, 10 iterasi (epoch); arsitektur disederhanakan (dua Dense 128), lr=0,01 dan batch=32 ditetapkan penulis. MobileNetV2 (transfer learning, bobot ImageNet dibekukan, Adam lr=1e-4) bukan bagian dari artikel tersebut. Augmentasi (rotasi 90, rotasi 270, flip vertikal, flip horizontal) hanya pada data latih setelah pembagian data; data validasi dan uji berupa citra asli. Tidak ada pemilihan epoch atau model berdasarkan data uji. Gunakan GPU (Runtime > Change runtime type > T4 GPU).

**Masukan:** Dataset (Mendeley Data): Palm doi:10.17632/ccr8cm22vz.1, Nail doi:10.17632/2xx4j3kjg2.1, Conjunctiva doi:10.17632/nt7r8hv2pz.1. Letakkan dalam folder `Dataset_Multisite/{Palm,Nail,Conjunctiva}`.

**Keluaran (folder `hasil_revisi`):** `baseline_cnn_palm_data_bersih.csv` (confusion matrix, akurasi, presisi, sensitivitas, spesifisitas, F1, AUC, persentase prediksi anemic), `riwayat_pelatihan_cnn.csv`, `kurva_pelatihan_cnn.png`.

**Kaitan dengan naskah:** Tabel 2 dan Lampiran A1 (baris AlexNet dan MobileNetV2) serta Tabel 10 (baris AlexNet).

**Cara menjalankan:** Colab, Runtime > Run all. CPU sudah cukup; GPU tidak diperlukan. Seed acak = 42 dan urutan file diurutkan (`sorted`) sehingga hasil dapat direproduksi pada versi pustaka yang dicetak di sel kedua.

In [1]:
!pip install lightgbm -q

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

Mounted at /content/drive


In [3]:
# ===== FUNGSI BERSAMA (identik di keempat notebook) =====
import os, re, hashlib, warnings
import numpy as np, pandas as pd, cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore')

# ---------- KONFIGURASI (ubah hanya bagian ini) ----------
SEED = 42
BASE = '/content/drive/MyDrive/Dataset_Multisite'   # folder berisi subfolder Palm/, Nail/, Conjunctiva/
OUT  = '/content/drive/MyDrive/hasil_revisi'          # folder keluaran (CSV)
SITES = {'Palm': 0, 'Nail': 1, 'Conjunctiva': 2}
# Filter piksel memeriksa kanal L, R, dan G. Kanal B ikut difilter jika True
# (Appiahene et al. 2023 menuliskan kondisi min < R=G=B < max).
FILTER_INCLUDE_B = False
# True: buang juga citra bernama varian dari citra lain (akhiran FV/RA atau spasi+angka) yang citra dasarnya ada.
BUANG_VARIAN = True
os.makedirs(OUT, exist_ok=True)

# ---------- DAFTAR FILE ----------
def parse_label(name):
    """Label dari awalan nama file (Anemic / Non-Anemic); nama yang mengandung tanda kurung (salinan bernomor) dan awalan tidak dikenali dilewati."""
    if '(' in name: return None
    c = name.lower().replace(' ', '')
    if c.startswith('non-anemic'): return 0
    if c.startswith('anemic'): return 1
    return None

def list_used_files():
    rows = []
    for site, sid in SITES.items():
        d = os.path.join(BASE, site)
        for n in sorted(os.listdir(d)):          # sorted -> urutan deterministik
            lab = parse_label(n)
            if lab is not None:
                rows.append(dict(site=site, site_id=sid, file=n, path=os.path.join(d, n), label=lab))
    return pd.DataFrame(rows)

def md5(path):
    with open(path, 'rb') as f: return hashlib.md5(f.read()).hexdigest()

# ---------- EKSTRAKSI FITUR ----------
def feat_original(img, lo=50, hi=200):
    """Fitur [mean a*, mean b*, mean G] dengan filter piksel. Return (fitur, mask_kosong)."""
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    L, a, b = cv2.split(lab); R, G, B = cv2.split(rgb)
    m = (L > lo) & (L < hi) & (R > lo) & (R < hi) & (G > lo) & (G < hi)
    if FILTER_INCLUDE_B: m &= (B > lo) & (B < hi)
    if m.any(): return [a[m].mean(), b[m].mean(), G[m].mean()], False
    return [0.0, 0.0, 0.0], True

def feat_otsu(img):
    """Fitur dari piksel foreground Otsu (grayscale). Tanpa filter L/R/G. Return (fitur, mask_kosong)."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mk = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = mk > 0
    if not m.any(): return [0.0, 0.0, 0.0], True
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    _, a, b = cv2.split(lab); G = rgb[:, :, 1]
    return [a[m].mean(), b[m].mean(), G[m].mean()], False

# ---------- PEMBAGIAN DATA & EVALUASI ----------
def split_idx(y, scheme, seed=SEED):
    """Return (train, val, test) berupa indeks. Skema: '80:20', '80:10:10', '70:10:20'."""
    idx = np.arange(len(y))
    if scheme == '80:20':
        tr, te = train_test_split(idx, test_size=0.20, stratify=y, random_state=seed)
        return tr, None, te
    if scheme == '80:10:10':
        tr, tmp = train_test_split(idx, test_size=0.20, stratify=y, random_state=seed)
        va, te = train_test_split(tmp, test_size=0.50, stratify=y[tmp], random_state=seed)
        return tr, va, te
    if scheme == '70:10:20':
        trva, te = train_test_split(idx, test_size=0.20, stratify=y, random_state=seed)
        tr, va = train_test_split(trva, test_size=0.125, stratify=y[trva], random_state=seed)
        return tr, va, te
    raise ValueError(scheme)

def metrics_from(yte, pred, proba):
    tn, fp, fn, tp = confusion_matrix(yte, pred, labels=[0, 1]).ravel()
    assert tn + fp + fn + tp == len(yte), 'TP+TN+FP+FN harus sama dengan jumlah data uji'
    two = len(set(yte)) == 2
    return dict(n_test=len(yte), n_test_anemic=int((yte == 1).sum()), TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                acc=100*(tp+tn)/len(yte),
                sens=100*tp/(tp+fn) if tp+fn else np.nan,
                spec=100*tn/(tn+fp) if tn+fp else np.nan,
                f1=100*f1_score(yte, pred, zero_division=0),
                auc=100*roc_auc_score(yte, proba) if two else np.nan,
                baseline_mayoritas=100*max((yte == 1).mean(), (yte == 0).mean()))

def fit_lgbm(Xtr, ytr):
    sc = StandardScaler().fit(Xtr)
    m = LGBMClassifier(n_estimators=100, random_state=SEED, verbose=-1).fit(sc.transform(Xtr), ytr)
    return sc, m

def predict_lgbm(sc, m, X):
    proba = m.predict_proba(sc.transform(X))[:, 1]
    return (proba >= 0.5).astype(int), proba

def tandai_dibuang(df):
    """Array boolean (urut baris): True bila baris dibuang pada pembersihan = salinan identik MD5 (satu dipertahankan),
    kedua file pada pasangan identik berlabel berbeda, dan (bila BUANG_VARIAN) citra varian FV/RA. df: indeks 0..n-1 dan kolom md5."""
    konflik = df.groupby('md5').label.transform('nunique') > 1
    dibuang = np.array(df.duplicated('md5', keep='first') | konflik, dtype=bool)   # salinan yang dapat diubah
    if BUANG_VARIAN:
        posisi = np.flatnonzero(~dibuang)
        for a, b in pasangan_varian(df.iloc[posisi].reset_index(drop=True)): dibuang[posisi[a]] = True
    return dibuang

def bersihkan(df):
    """Data bersih untuk semua eksperimen (satu definisi tunggal)."""
    df = df.reset_index(drop=True).copy(); df['md5'] = df['path'].map(md5)
    return df[~tandai_dibuang(df)].reset_index(drop=True)

def cek_kebocoran(md5s, tr, va, te, wajib=True):
    """Pemeriksaan kebocoran: indeks latih/validasi/uji harus saling lepas dan tidak ada citra identik (MD5) lintas partisi.
    Return jumlah citra identik lintas partisi, mis. {'latih-uji': 0}. wajib=True -> hentikan program bila ada pelanggaran."""
    parts = {'latih': set(tr), 'uji': set(te)}
    if va is not None: parts['val'] = set(va)
    nm = sorted(parts)
    for i, a in enumerate(nm):
        for b in nm[i + 1:]:
            assert not (parts[a] & parts[b]), f'Indeks tumpang tindih: {a}-{b}'
    m = {k: set(np.asarray(md5s)[sorted(v)]) for k, v in parts.items()}
    bocor = {f'{a}-{b}': len(m[a] & m[b]) for i, a in enumerate(nm) for b in nm[i + 1:]}
    if wajib: assert all(v == 0 for v in bocor.values()), f'Citra identik lintas partisi: {bocor}'
    return bocor

def pasangan_varian(df):
    """Pasangan (indeks varian, indeks citra dasar) pada df (indeks posisi 0..n-1): nama berakhiran FV/RA atau spasi+angka
    yang nama dasarnya ada pada situs yang sama (mis. Anemic-001FV dengan Anemic-001)."""
    stem = df.file.str.replace(r'\.[A-Za-z]+$', '', regex=True)
    base = stem.str.replace(r'\s\d+$', '', regex=True).str.replace(r'(FV|RA)$', '', regex=True)
    key = {(s, t): i for i, (s, t) in enumerate(zip(df.site, stem))}
    return [(i, key[(s, b)]) for i, (s, t, b) in enumerate(zip(df.site, stem, base)) if b != t and (s, b) in key]

def cek_varian(pasangan, tr, va, te):
    """Jumlah pasangan varian-dasar yang terpisah di partisi berbeda (informasi; idealnya 0)."""
    part = {}
    for ids, nm in [(tr, 'latih'), (va, 'val'), (te, 'uji')]:
        if ids is not None:
            for i in ids: part[i] = nm
    return sum(1 for a, b in pasangan if a in part and b in part and part[a] != part[b])

In [4]:
import sys, numpy, pandas, cv2, sklearn, lightgbm
print('Python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| OpenCV', cv2.__version__, '| scikit-learn', sklearn.__version__, '| LightGBM', lightgbm.__version__, '| SEED', SEED)

Python 3.13.16 | numpy 2.4.6 | pandas 2.2.3 | OpenCV 4.14.0 | scikit-learn 1.6.1 | LightGBM 4.6.0 | SEED 42


In [5]:
# ===== BASELINE CNN PADA DATA PALM BERSIH: AlexNet (mengikuti [1]) dan MobileNetV2 (tambahan penulis) =====
import tensorflow as tf
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, optimizers, regularizers

# ---------- KONFIGURASI ----------
SCHEMES = ['70:10:20', '80:10:10']
IMG, BATCH, EPOCHS = 224, 32, 10       # EPOCHS=10: batas iterasi maksimum pada [1]; ukuran input dan batch ditetapkan penulis
AUGMENT = True                         # rotasi 90, rotasi 270, flip vertikal, flip horizontal: HANYA pada data latih, setelah split
MN_PRETRAINED = 'imagenet'             # bobot ImageNet untuk MobileNetV2 (perlu internet di Colab)
ALEXNET_MOMENTUM = 0.0                 # 0,0 = SGD biasa seperti [1]; ubah (mis. 0,9) hanya sebagai varian yang diberi label modifikasi penulis
DETERMINISTIK = True                   # operasi deterministik; bila muncul error 'determinism', ubah ke False lalu restart runtime

tf.keras.utils.set_random_seed(SEED)
if DETERMINISTIK:
    try: tf.config.experimental.enable_op_determinism()
    except Exception as e: print('Determinisme operasi tidak aktif:', e)
print('GPU:', tf.config.list_physical_devices('GPU'))

# ---------- DATA: Palm bersih, citra dipakai apa adanya (tanpa statistik yang dipelajari dari seluruh data) ----------
palm = bersihkan(list_used_files()); palm = palm[palm.site == 'Palm'].reset_index(drop=True)
y = palm.label.to_numpy()
def prep(path):
    im = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    return cv2.resize(im, (IMG, IMG), interpolation=cv2.INTER_AREA)
IM = np.stack([prep(p) for p in palm.path]).astype('uint8')
print('Citra Palm bersih:', len(y), '| anemic:', int(y.sum()), '| non-anemic:', int((y == 0).sum()), '| bentuk:', IM.shape)

def versi(x):   # asli, rotasi 90 searah jarum jam, rotasi 270, flip vertikal, flip horizontal
    return [x, np.rot90(x, -1), np.rot90(x, 1), x[::-1], x[:, ::-1]]

# ---------- MODEL ----------
def build_alexnet():
    """AlexNet: 5 konvolusi + 3 max-pooling, ReLU, regularisasi L2 0,0001, SGD ([1]). Dua lapisan Dense 128 + Dropout 0,5
    (bukan 4096 seperti AlexNet asli karena data kecil) dan lr=0,01 ditetapkan penulis."""
    reg = regularizers.l2(1e-4)
    m = models.Sequential([
        layers.Input((IMG, IMG, 3)), layers.Rescaling(1. / 255),
        layers.Conv2D(96, (11, 11), strides=4, activation='relu', kernel_regularizer=reg), layers.MaxPooling2D((3, 3), strides=2),
        layers.Conv2D(256, (5, 5), padding='same', activation='relu', kernel_regularizer=reg), layers.MaxPooling2D((3, 3), strides=2),
        layers.Conv2D(384, (3, 3), padding='same', activation='relu', kernel_regularizer=reg),
        layers.Conv2D(384, (3, 3), padding='same', activation='relu', kernel_regularizer=reg),
        layers.Conv2D(256, (3, 3), padding='same', activation='relu', kernel_regularizer=reg), layers.MaxPooling2D((3, 3), strides=2),
        layers.Flatten(), layers.Dense(128, activation='relu', kernel_regularizer=reg), layers.Dropout(0.5),
        layers.Dense(128, activation='relu', kernel_regularizer=reg), layers.Dropout(0.5), layers.Dense(1, activation='sigmoid')])
    m.compile(optimizer=optimizers.SGD(learning_rate=0.01, momentum=ALEXNET_MOMENTUM), loss='binary_crossentropy', metrics=['accuracy'])
    return m

def build_mobilenet():
    """MobileNetV2 transfer learning (bukan dari [1]): bobot ImageNet dibekukan, kepala GAP + Dropout 0,2 + Dense; Adam lr=1e-4."""
    base = tf.keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights=MN_PRETRAINED)
    base.trainable = False
    inp = layers.Input((IMG, IMG, 3))
    x = layers.Rescaling(1. / 127.5, offset=-1)(inp)                   # praproses MobileNetV2: [0,255] -> [-1,1]
    x = base(x, training=False)
    x = layers.Dropout(0.2)(layers.GlobalAveragePooling2D()(x))
    m = models.Model(inp, layers.Dense(1, activation='sigmoid')(x))
    m.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return m

KONFIG = {'AlexNet': (build_alexnet, f'SGD lr=0,01 momentum={ALEXNET_MOMENTUM}, L2=0,0001, ReLU, {EPOCHS} epoch [1]'),
          'MobileNetV2 (transfer learning)': (build_mobilenet, f'Adam lr=1e-4, ImageNet beku, {EPOCHS} epoch')}

rows, riwayat = [], []
for scheme in SCHEMES:
    tr, va, te = split_idx(y, scheme)
    bocor = cek_kebocoran(palm.md5.to_numpy(), tr, va, te)
    vlin = cek_varian(pasangan_varian(palm), tr, va, te)
    k = 5 if AUGMENT else 1
    Xtr = np.stack([np.ascontiguousarray(v) for i in tr for v in versi(IM[i])[:k]]); ytr = np.repeat(y[tr], k); grp = np.repeat(tr, k)
    assert not (set(grp) & set(te)) and not (set(grp) & set(va)), 'Salinan augmentasi citra uji/validasi masuk ke data latih'
    print(f'\nSkema {scheme}: pemeriksaan kebocoran lolos {bocor}, pasangan varian lintas partisi={vlin}; citra latih={len(tr)} (baris={len(ytr)}), val={len(va)}, uji={len(te)}')
    for nama, (builder, konf) in KONFIG.items():
        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)       # seed sama untuk setiap model dan skema
        m = builder()
        try:
            h = m.fit(Xtr, ytr, batch_size=BATCH, epochs=EPOCHS, validation_data=(IM[va], y[va]), shuffle=True, verbose=0)
        except Exception as e:
            if 'determin' in str(e).lower(): print('Operasi tidak mendukung mode deterministik: set DETERMINISTIK = False, restart runtime, jalankan ulang.')
            raise
        pb = m.predict(IM[te], batch_size=BATCH, verbose=0).ravel(); vb = m.predict(IM[va], batch_size=BATCH, verbose=0).ravel()
        pr = (pb >= 0.5).astype(int)
        r = metrics_from(y[te], pr, pb); v = metrics_from(y[va], (vb >= 0.5).astype(int), vb)
        prec = 100 * r['TP'] / (r['TP'] + r['FP']) if (r['TP'] + r['FP']) else np.nan
        rows.append(dict(skema=scheme, algoritma=nama, konfigurasi=konf, n_latih_baris=len(ytr), n_val=len(va), **r, precision=prec,
                         pct_prediksi_anemic=100 * pr.mean(), val_acc=v['acc'], val_auc=v['auc'], parameter=m.count_params()))
        for e in range(len(h.history['loss'])):
            riwayat.append(dict(skema=scheme, algoritma=nama, epoch=e + 1, **{kk: vv[e] for kk, vv in h.history.items()}))
        print(f'  {nama}: acc={r["acc"]:.2f} auc={r["auc"]:.2f} sens={r["sens"]:.2f} spec={r["spec"]:.2f} | TP={r["TP"]} TN={r["TN"]} FP={r["FP"]} FN={r["FN"]} | diprediksi anemic={100*pr.mean():.1f}%')

hasil = pd.DataFrame(rows)
kolom = ['skema','algoritma','konfigurasi','n_latih_baris','n_val','n_test','n_test_anemic','TP','TN','FP','FN','acc','precision','sens','spec','f1','auc',
         'val_acc','val_auc','baseline_mayoritas','pct_prediksi_anemic','parameter']
hasil = hasil[kolom].round(2)
hasil.to_csv(os.path.join(OUT, 'baseline_cnn_palm_data_bersih.csv'), index=False)
rw = pd.DataFrame(riwayat).round(4); rw.to_csv(os.path.join(OUT, 'riwayat_pelatihan_cnn.csv'), index=False)
print('\n', hasil[['skema','algoritma','acc','precision','sens','spec','f1','auc','val_acc','val_auc','baseline_mayoritas','pct_prediksi_anemic']].to_string(index=False))

# kurva pelatihan (bukti ada/tidaknya kolaps ke kelas mayoritas)
fig, ax = plt.subplots(len(SCHEMES), 2, figsize=(11, 4 * len(SCHEMES)), squeeze=False)
for i, scheme in enumerate(SCHEMES):
    for j, nama in enumerate(KONFIG):
        d = rw[(rw.skema == scheme) & (rw.algoritma == nama)]
        ax[i][j].plot(d.epoch, d['loss'], label='loss latih'); ax[i][j].plot(d.epoch, d['val_loss'], label='loss validasi')
        ax[i][j].set_title(f'{nama} ({scheme})'); ax[i][j].set_xlabel('epoch'); ax[i][j].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'kurva_pelatihan_cnn.png'), dpi=150)

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Citra Palm bersih: 519 | anemic: 301 | non-anemic: 218 | bentuk: (519, 224, 224, 3)

Skema 70:10:20: pemeriksaan kebocoran lolos {'latih-uji': 0, 'latih-val': 0, 'uji-val': 0}, pasangan varian lintas partisi=0; citra latih=363 (baris=1815), val=52, uji=104
  AlexNet: acc=57.69 auc=53.18 sens=100.00 spec=0.00 | TP=60 TN=0 FP=44 FN=0 | diprediksi anemic=100.0%
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
  MobileNetV2 (transfer learning): acc=65.38 auc=70.49 sens=81.67 spec=43.18 | TP=49 TN=19 FP=25 FN=11 | diprediksi anemic=71.2%

Skema 80:10:10: pemeriksaan kebocoran lolos {'latih-uji': 0, 'latih-val': 0, 'uji-val': 0}, pasangan varian lintas partisi=0; citra latih=415 (baris=2075), val=52, uji=52


  AlexNet: acc=57.69 auc=60.30 sens=100.00 spec=0.00 | TP=30 TN=0 FP=22 FN=0 | diprediksi anemic=100.0%
  MobileNetV2 (transfer learning): acc=59.62 auc=71.52 sens=73.33 spec=40.91 | TP=22 TN=9 FP=13 FN=8 | diprediksi anemic=67.3%

    skema                       algoritma   acc  precision   sens  spec    f1   auc  val_acc  val_auc  baseline_mayoritas  pct_prediksi_anemic
70:10:20                         AlexNet 57.69      57.69 100.00  0.00 73.17 53.18    57.69    55.76               57.69               100.00
70:10:20 MobileNetV2 (transfer learning) 65.38      66.22  81.67 43.18 73.13 70.49    57.69    65.45               57.69                71.15
80:10:10                         AlexNet 57.69      57.69 100.00  0.00 73.17 60.30    57.69    46.97               57.69               100.00
80:10:10 MobileNetV2 (transfer learning) 59.62      62.86  73.33 40.91 67.69 71.52    67.31    70.76               57.69                67.31


In [6]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
OUT = '/content/drive/MyDrive/hasil_revisi'          # folder berisi CSV hasil
LABEL = ['Non-anemic', 'Anemic']

def cm_dari_baris(r):
    """Matriks [[TN, FP], [FN, TP]]: baris = kelas sebenarnya, kolom = prediksi."""
    cm = np.array([[r['TN'], r['FP']], [r['FN'], r['TP']]], dtype=int)
    n = r['n_test'] if 'n_test' in r.index else r['n']
    assert cm.sum() == n, f'TP+TN+FP+FN ({cm.sum()}) harus sama dengan jumlah data ({n})'
    return cm

def gambar_cm(cm, judul, ax):
    pct = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None) * 100      # persen per kelas sebenarnya
    ax.imshow(pct, cmap='Blues', vmin=0, vmax=100)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{cm[i, j]}\n({pct[i, j]:.1f}%)', ha='center', va='center', fontsize=9,
                    color='white' if pct[i, j] > 50 else 'black')
    ax.set_xticks([0, 1]); ax.set_xticklabels(LABEL, fontsize=8); ax.set_yticks([0, 1]); ax.set_yticklabels(LABEL, fontsize=8)
    ax.set_xlabel('Prediksi'); ax.set_ylabel('Sebenarnya'); ax.set_title(judul, fontsize=8)

def grid_cm(df, judul, ncol=4, simpan=None, judul_utama=''):
    n = len(df); nrow = int(np.ceil(n / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(3.3 * ncol, 3.4 * nrow), squeeze=False)
    for ax in axs.ravel(): ax.axis('off')
    for ax, (_, r) in zip(axs.ravel(), df.iterrows()):
        ax.axis('on'); gambar_cm(cm_dari_baris(r), judul(r), ax)
    fig.suptitle(judul_utama); plt.tight_layout()
    if simpan: plt.savefig(os.path.join(OUT, simpan), dpi=150)
    plt.show()

In [7]:
c = pd.read_csv(f'{OUT}/baseline_cnn_palm_data_bersih.csv')
grid_cm(c, lambda r: f"{r['algoritma'][:22]} ({r['skema']})", ncol=4, simpan='cm_nb21.png', judul_utama='Notebook 21')

v = pd.read_csv(f'{OUT}/varian_cnn_palm_data_bersih.csv')
grid_cm(v, lambda r: f"{r['algoritma'][:22]} ({r['skema']})", ncol=3, simpan='cm_nb22.png', judul_utama='Notebook 22')